# More Than A Hack 2027

**Ziel:** Die vorhandenen Eingabedaten des Spiels Barcelona–Athletic kennenlernen, ihre Zeitachsen verbinden und wichtige Qualitätsprobleme sichtbar machen.

Wir untersuchen Dateien, Metadaten, Tracking, Balllücken, taktische Tags und die Projektion ins taktische Video. Dieses Notebook erzeugt **keine Ballbesitzlabels**, trainiert kein Modell und korrigiert keine Rohdaten.

**Ausführung:** Python-Kernel mit `numpy`, `pandas`, `matplotlib`, `opencv-python` und `IPython` wählen, dann **Restart Kernel → Run All**. Falls nötig, diese Pakete zuvor in der eigenen Projektumgebung installieren. Es werden keine Pakete automatisch installiert und keine Daten heruntergeladen. Der Videodecoder liest nur einzelne Bilder, nicht das gesamte Video in den Arbeitsspeicher.

**Vertraulichkeit:** Ausgaben enthalten Trackingauszüge und Videobilder und sind nur für die lokale Arbeit bestimmt. Vor einem öffentlichen Commit die Notebook-Ausgaben entfernen und die Matchdateien ausschließen.

Referenzen: [Daten und Zeitachsen](https://github.com/metrica-sports/more-than-a-hack-2027/blob/main/DATA.md), [Aufgabe](https://github.com/metrica-sports/more-than-a-hack-2027/blob/main/CHALLENGE.md), [Bewertung](https://github.com/metrica-sports/more-than-a-hack-2027/blob/main/EVALUATION.md), [Nutzungsregeln](https://github.com/metrica-sports/more-than-a-hack-2027/blob/main/RULES.md).

In [ ]:
# Standardbibliothek
from pathlib import Path
from importlib.metadata import version
import sys
import xml.etree.ElementTree as ET

# Daten und Video
import numpy as np
import pandas as pd
import cv2

# Darstellung
import matplotlib.pyplot as plt
from matplotlib.patches import Circle, Rectangle
from IPython.display import display, Markdown

## 1. Konfiguration und Dateien

Der Projektordner wird anhand der tatsächlich vorhandenen taktischen XML gefunden; dadurch funktioniert das Notebook auch aus seinem Unterordner. Die folgenden Halbzeitgrenzen sind **veröffentlichte Evaluationsbereiche**, keine abgeleiteten Ballbesitzlabels. Sie dienen dazu, Vorlauf, Halbzeitpause und Nachlauf getrennt zu betrachten. Die Zeitangaben beziehen sich auf das Video, nicht auf die Stadion-Spieluhr.

Quelle der Grenzen: [DATA.md, Which frames are graded](https://github.com/metrica-sports/more-than-a-hack-2027/blob/main/DATA.md#which-frames-are-graded). Die Roh-XML trägt diese beiden Halbzeitbereiche nicht korrekt getrennt ein; wir verwenden ihre Periodenangaben deshalb nicht als Spieluhr.

In [ ]:
PROJECT_DIR = next(p for p in (Path.cwd(), *Path.cwd().parents)
                   if (p / "tactical/BAR-ATH_tactical_atd.xml").is_file())

paths = {
    "metadata": PROJECT_DIR / "tactical/BAR-ATH_tactical_atd.xml",
    "tracking": PROJECT_DIR / "tactical/BAR-ATH_tactical_atd.txt",
    "homography": PROJECT_DIR / "tactical/BAR-ATH_tactical_homography.csv",
    "tags": PROJECT_DIR / "tactical/BAR-ATH_tactical_smart_tagging.xml",
    "video": PROJECT_DIR / "tactical/BAR-ATH_tactical.mp4",
}

HALVES = [("1. Halbzeit", 11185, 90651), ("2. Halbzeit", 110808, 190197)]
TEAM_COLORS = {"H": "#1764ab", "A": "#c83e41"}

plt.rcParams.update({"figure.dpi": 110, "axes.spines.top": False, "axes.spines.right": False})
pd.set_option("display.max_columns", 12)

In [ ]:
assets = pd.DataFrame([
    {"Quelle": source, "Datei": p.name, "Größe (MB)": p.stat().st_size / 1_000_000}
    for source in ["tactical", "broadcast"]
    for p in sorted((PROJECT_DIR / source).glob("*")) if not p.name.startswith("._")
])

display(assets.round(2))

**Einordnung:** Tactical ist die Referenzzeitachse für die Abgabe. Broadcast wird hier nur im Dateiinventar aufgeführt; seine Aufnahmen und Trackingdaten sind nicht mit Tactical synchronisiert und werden nicht vermischt. Die vorhandene `README.md` beschreibt den Datenunterordner des Veranstalter-Repositories; sie ist nicht die vollständige Aufgabenstellung.

## 2. Was beschreibt die Tracking-XML?

Die XML enthält Teams, Track-Slots, Spielfeldmaße, Abtastrate und die Reihenfolge der Kanäle im Textformat. Die Teamzuordnung `H = Barcelona`, `A = Athletic` stammt aus der offiziellen Aufgabenstellung. Ein Track-Slot ist **keine dauerhafte Spieleridentität**; die Nummern sind keine verlässlichen Trikotnummern.

In [ ]:
root = ET.parse(paths["metadata"]).getroot()
spec = root.find("./DataFormatSpecifications/DataFormatSpecification")
fps = float(root.findtext("./Metadata/GlobalConfig/FrameRate"))
n_frames = int(spec.get("endFrame")) - int(spec.get("startFrame")) + 1
field = root.find("./Metadata/Sessions/Session/FieldSize")
pitch_length, pitch_width = float(field.findtext("Width")), float(field.findtext("Height"))

teams = pd.DataFrame([{"team_id": t.get("id"), "team": t.findtext("Name")}
                      for t in root.findall("./Metadata/Teams/Team")])
teams["side"] = teams["team"].map({"Barcelona": "H", "Atletic Club": "A"})
tracks = pd.DataFrame([{"track_id": p.get("id"), "track_name": p.findtext("Name"), "team_id": p.get("teamId")}
                       for p in root.findall("./Metadata/Players/Player")])
tracks = tracks.merge(teams, on="team_id", validate="many_to_one")
channel_to_track = {c.get("id"): c.get("playerId") for c in root.findall("./Metadata/PlayerChannels/PlayerChannel")}
track_ids = [channel_to_track[g.find("PlayerChannelRef").get("playerChannelId")]
             for g in spec.findall("./SplitRegister/SplitRegister")]
tracks = tracks.set_index("track_id").loc[track_ids]

print(f"{n_frames:,} Frames | {fps:g} fps | Spielfeld laut XML: {pitch_length:g} × {pitch_width:g} m")
display(tracks.groupby(["side", "team"], sort=False).size().rename("Track-Slots").to_frame())
display(tracks[["track_name", "team", "side"]].head())

In [ ]:
channels = pd.DataFrame([
    {"Kanal": c.get("id"), "Bezeichnung": c.findtext("Name"), "Einheit laut XML": c.findtext("Unit")}
    for c in root.findall("./Metadata/Devices/Device/Sensors/Sensor/Channels/Channel")])

display(channels)

**Wichtige Unstimmigkeit:** Der rohe dritte Kanal `s` ist in dieser XML als `speed_kph` mit Einheit `kph` beschrieben. Die offizielle Dokumentation beschreibt die Ausgabe des Loaders dagegen in m/s. Da wir die Textdatei direkt lesen, nennen wir den Kanal **`speed_raw`** und wandeln ihn nicht stillschweigend um. Für dieses Einstiegsnotebook werden daraus keine Bewegungsmerkmale berechnet.

Die Positionskanäle sind auf 0–1 normalisiert. Für die räumlichen Plots verwenden wir die lokalen XML-Maße 105 × 68 m. Das Veranstalter-Repository nennt zusätzlich 105,3 × 68 m aus korrigiertem Tracking; eine spätere metrische Besitzregel muss diese Wahl ausdrücklich festhalten. Hier beeinflusst sie keine Klassifikation.

## 3. Rohtracking laden und lesbar machen

Jede Zeile hat die Form `frame_id:Spielergruppen:Ballgruppe`. Jede Gruppe enthält `x,y,s`; fehlende Werte sind `NaN`. Wir lesen alle Frames unverändert ein und ordnen die Spielergruppen anhand der XML zu. `float32` begrenzt den Speicherbedarf; die ursprünglichen Textdateien bleiben unverändert.

Die vollständige Tabelle `tracking` hat zweistufige Spalten: Objekt/Track und Kanal, zum Beispiel `tracking[("ball", "x")]`. `frame` ist ab hier immer der **Videoindex ab 0**. Eine Prüfung verhindert, dass fehlende oder vertauschte Rohzeilen unbemerkt zu falscher Videoausrichtung führen.

In [ ]:
values = np.empty((n_frames, len(track_ids) + 1, 3), dtype=np.float32)
frame_ids = np.empty(n_frames, dtype=np.int64)
with paths["tracking"].open() as handle:
    for row, line in enumerate(handle):
        frame_id, players_raw, ball_raw = line.strip().split(":")
        frame_ids[row] = int(frame_id)
        values[row] = np.fromstring(players_raw.replace(";", ",") + "," + ball_raw, sep=",", dtype=np.float32).reshape(-1, 3)

assert row + 1 == n_frames and np.array_equal(frame_ids, np.arange(1, n_frames + 1)), "Tracking-Zeitachse passt nicht zum erwarteten Videoindex."
tracking = pd.DataFrame(values.reshape(n_frames, -1), index=pd.Index(frame_ids - 1, name="frame"),
                        columns=pd.MultiIndex.from_product([track_ids + ["ball"], ["x", "y", "speed_raw"]],names=["object", "channel"]))
print(f"Tracking: {tracking.shape[0]:,} Zeilen × {tracking.shape[1]} Kanäle; {values.nbytes / 1e6:.1f} MB numerische Rohwerte")
display(tracking.loc[:, pd.IndexSlice[[track_ids[0], track_ids[-1], "ball"], :]].head())

## 4. Video und Homographien auf dieselbe Zeitachse bringen

Eine Homographie ist eine 3×3-Matrix, die **normalisierte Spielfeldpositionen auf Videopixel** abbildet. Sie wird erst zur visuellen Projektion benötigt; Entfernungen auf dem Spielfeld benötigen sie nicht. `valid` und `reprojection_error` beschreiben die Verfügbarkeit und Güte der Kamerakalibrierung, nicht die Qualität des Balltrackings.

In [ ]:
cap = cv2.VideoCapture(str(paths["video"]))
if not cap.isOpened():
    raise RuntimeError(f"Video kann nicht geöffnet werden: {paths['video']}")
video_info = {"Frames": int(cap.get(cv2.CAP_PROP_FRAME_COUNT)), "fps": cap.get(cv2.CAP_PROP_FPS),
              "Breite (px)": int(cap.get(cv2.CAP_PROP_FRAME_WIDTH)), "Höhe (px)": int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))}
cap.release()
homography = pd.read_csv(paths["homography"]).set_index("frame")
assert video_info["Frames"] == n_frames and np.isclose(video_info["fps"], fps), "Video und Tracking haben unterschiedliche Zeitachsen."
assert np.array_equal(homography.index, tracking.index), "Homographien und Tracking sind nicht framegleich."
h_columns = [f"h{i}{j}" for i in range(3) for j in range(3)]

print(video_info)
display(pd.DataFrame([
    {"Quelle": "ATD roh", "Originalindex": "1 … 199319", "Umrechnung": "frame_id − 1"},
    {"Quelle": "Video / Homographie", "Originalindex": "0 … 199318", "Umrechnung": "keine"},
    {"Quelle": "Smart Tags", "Originalindex": "Videosekunden", "Umrechnung": "round(Sekunden × fps)"},
]))

## 5. Wie vollständig sind die Beobachtungen?

Eine Position gilt als beobachtet, wenn **beide Koordinaten endlich** sind. Das sagt noch nichts über ihre Richtigkeit. Wir unterscheiden das gesamte Video von den veröffentlichten Evaluationsbereichen. Fehlende Positionen werden weder aufgefüllt noch als Spielunterbrechung interpretiert.

In [ ]:
player_xy, ball_xy = values[:, :-1, :2], values[:, -1, :2]
player_seen = np.isfinite(player_xy).all(axis=2)
ball_seen = np.isfinite(ball_xy).all(axis=1)
in_evaluation = np.zeros(n_frames, dtype=bool)
for _, first, last in HALVES:
    in_evaluation[first:last + 1] = True

frames = pd.DataFrame({"video_s": tracking.index.to_numpy() / fps, "ball_x": ball_xy[:, 0], "ball_y": ball_xy[:, 1],
                       "ball_observed": ball_seen, "n_tracks": player_seen.sum(axis=1),
                       "in_evaluation_window": in_evaluation}, index=tracking.index)
for side in ["H", "A"]:
    frames[f"n_{side}"] = player_seen[:, tracks["side"].eq(side).to_numpy()].sum(axis=1)

quality = pd.DataFrame({name: {
    "Frames": int(mask.sum()), "Ball beobachtet (%)": 100 * ball_seen[mask].mean(),
    "Track-Slots pro Frame (Mittel)": player_seen[mask].sum(axis=1).mean(),
    "Frames mit >22 Track-Slots (%)": 100 * (player_seen[mask].sum(axis=1) > 22).mean(),
    "Homographie valid (%)": 100 * homography.loc[mask, "valid"].eq(1).mean(),
} for name, mask in {"Gesamtes Video": np.ones(n_frames, dtype=bool), "Evaluationsbereiche": in_evaluation}.items()}).T
display(quality.round(2))
display(frames.loc[in_evaluation].head())

In [ ]:
coordinate_quality = []
for name, xy in [("Spieler-Slots", player_xy), ("Ball", ball_xy)]:
    complete = np.isfinite(xy).all(axis=-1)
    coordinate_quality.append({"Objekt": name, "x min": np.nanmin(xy[..., 0]), "x max": np.nanmax(xy[..., 0]),
                               "y min": np.nanmin(xy[..., 1]), "y max": np.nanmax(xy[..., 1]),
                               "Positionen außerhalb [0,1]": int((complete & ((xy < 0) | (xy > 1)).any(axis=-1)).sum()),
                               "Nur eine Koordinate vorhanden": int(np.isfinite(xy).any(axis=-1).sum() - complete.sum())})
display(pd.DataFrame(coordinate_quality).round(4))
print(f"Roh-Frame-IDs doppelt: {pd.Index(frame_ids).duplicated().sum()}")
print(f"Homographien ohne endliche Matrix: {(~np.isfinite(homography[h_columns]).all(axis=1)).sum()}")

**Interpretation:** Mehr als 22 beobachtete Slots können unter anderem durch doppelte Tracks entstehen; weniger als 22 durch fehlende Erfassung. Die Slot-Anzahl ist keine gesicherte Anzahl realer Spieler. Positionen außerhalb des Spielfeldrechtecks werden angezeigt und bleiben erhalten: Sie können einen Ball im Aus oder Messfehler beschreiben und dürfen nicht pauschal entfernt werden.

In [ ]:
minute_summary = frames.groupby((frames["video_s"] // 60).astype(int)).agg(
    ball_visible=("ball_observed", "mean"), home_tracks=("n_H", "mean"), away_tracks=("n_A", "mean"))
minute_x = minute_summary.index.to_numpy() + 0.5
fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True, constrained_layout=True)
axes[0].plot(minute_x, 100 * minute_summary["ball_visible"], color="#1764ab")
axes[0].set(title="Wie verändert sich die Tracking-Abdeckung im Video?", ylabel="Ball beobachtet (%)", ylim=(0, 105))
for side, column in [("H", "home_tracks"), ("A", "away_tracks")]:
    axes[1].plot(minute_x, minute_summary[column], label=teams.set_index("side").loc[side, "team"], color=TEAM_COLORS[side])
axes[1].axhline(11, color="0.5", linestyle=":", linewidth=1, label="11 Spieler je Team (Referenz)")
axes[1].set(xlabel="Videominute (keine Spieluhr)", ylabel="Beobachtete Slots\n(Mittel pro Minute)")
axes[1].legend(loc="upper right")
for ax in axes:
    for half, first, last in HALVES:
        ax.axvspan(first / fps / 60, (last + 1) / fps / 60, color="#2e8b57", alpha=0.07)
    ax.grid(alpha=0.2)
plt.show()
display(Markdown(f"**Beobachtung:** In den Evaluationsbereichen fehlt der Ball in **{100 * (~ball_seen[in_evaluation]).mean():.1f} %** der Frames. Die grün hinterlegten Bereiche sind die zwei bewerteten Halbzeiten. Eine spätere Besitzlogik muss auch ohne aktuelle Ballposition funktionieren."))

## 6. Wie lange fehlt der Ball am Stück?

Einzelne fehlende Frames und lange Ausfälle benötigen später unterschiedliche Behandlung. Wir messen zusammenhängende Lücken unverändert auf der Videozeitachse. Die Tabelle zeigt die längsten Lücken, die vollständig innerhalb einer bewerteten Halbzeit liegen; Vorlauf und Halbzeitpause werden dafür nicht mit Spielphasen vermischt.

In [ ]:
missing = ~ball_seen
starts = np.flatnonzero(missing & ~np.r_[False, missing[:-1]])
ends = np.flatnonzero(missing & ~np.r_[missing[1:], False])
ball_gaps = pd.DataFrame({"start_frame": starts, "end_frame": ends, "duration_s": (ends - starts + 1) / fps})
ball_gaps["half"] = pd.Series(index=ball_gaps.index, dtype="object")

for half, first, last in HALVES:
    ball_gaps.loc[(starts >= first) & (ends <= last), "half"] = half
play_gaps = ball_gaps.loc[ball_gaps["half"].notna()]
display(play_gaps.nlargest(8, "duration_s").assign(start_video_min=lambda d: d["start_frame"] / fps / 60).round(2))

fig, ax = plt.subplots(figsize=(9, 3.5), constrained_layout=True)
ax.hist(play_gaps["duration_s"], bins=np.geomspace(1 / fps, max(play_gaps["duration_s"].max(), 2 / fps), 35), color="#1764ab")
ax.set(xscale="log", title="Balllücken innerhalb der bewerteten Halbzeiten", xlabel="Dauer einer Lücke (Sekunden, logarithmisch)", ylabel="Anzahl Lücken")
ax.grid(axis="y", alpha=0.2)
plt.show()
display(Markdown(f"**Beobachtung:** Es gibt **{len(play_gaps):,}** vollständig in einer Halbzeit liegende Balllücken. Median: **{play_gaps['duration_s'].median():.2f} s**, längste: **{play_gaps['duration_s'].max():.2f} s**. Die Ursache (Verdeckung, Ball im Aus oder Trackingfehler) lässt sich aus der Lückendauer allein nicht bestimmen."))

## 7. Was liefern die Smart Tags?

Die SportsCode-XML beschreibt automatische taktische Phasen und Standards als Intervalle in **Videosekunden**. Wir übernehmen die vorhandenen Taggruppen unverändert, zeigen Häufigkeiten und rechnen die Zeiten in Frameindizes um. Fehlende Tags bleiben fehlend. Intervalle können sich überlappen und sind keine fertigen Ballbesitzphasen.

In [ ]:
tag_root = ET.parse(paths["tags"]).getroot()
events = pd.DataFrame([
    {"id": int(e.findtext("ID")), "start_s": float(e.findtext("start")), "end_s": float(e.findtext("end")),
     "code": e.findtext("code"), **{label.findtext("group"): label.findtext("text") for label in e.findall("label")}}
    for e in tag_root.findall("./ALL_INSTANCES/instance")
])

events["start_frame"] = np.rint(events["start_s"] * fps).astype(int)
events["end_frame"] = np.rint(events["end_s"] * fps).astype(int)
events["duration_s"] = events["end_s"] - events["start_s"]
display(events[["id", "code", "Team", "Type", "Half", "start_s", "end_s", "start_frame"]].head(8))
print(f"{len(events)} Intervalle; doppelte IDs: {events['id'].duplicated().sum()}; nichtpositive Dauer: {events['duration_s'].le(0).sum()}")
print(f"Intervalle außerhalb der Videodauer: {((events['start_s'] < 0) | (events['end_s'] > n_frames / fps)).sum()}")
display(events["Team"].value_counts(dropna=False).rename("Intervalle").to_frame())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5), constrained_layout=True)
events["code"].value_counts().sort_values().plot.barh(ax=axes[0], color="#1764ab")
axes[0].set(title="Welche taktischen Phasen sind vorhanden?", xlabel="Anzahl Intervalle", ylabel="")
events.loc[events["code"].eq("SET PIECES"), "Type"].value_counts().sort_values().plot.barh(ax=axes[1], color="#8c6bb1")
axes[1].set(title="Welche Standards sind getaggt?", xlabel="Anzahl Intervalle", ylabel="")
plt.show()
display(Markdown(f"**Beobachtung:** Die Tags enthalten **{events['code'].nunique()}** Phasentypen und **{events['Team'].eq('N/A').sum()}** Intervalle mit Team `N/A`. Ein Team-Tag muss im Zusammenhang mit dem Phasentyp gelesen werden: Eine Defensivphase bezeichnet nicht automatisch das ballbesitzende Team. Standardintervalle sind keine exakt validierten Dead-Ball-Grenzen."))

## 8. Eine Szene im Tracking und im Video ansehen

Wir wählen deterministisch den ersten bewerteten Frame mit beobachtetem Ball und mindestens 20 beobachteten Spieler-Slots. Das ist eine **anschauliche Einzelszene**, keine repräsentative Stichprobe. `example_frame` kann später gezielt geändert werden; die folgenden Zellen zeigen dazu die Szene und die aktiven Tags.

Links sehen wir metrische Spielfeldpositionen, rechts dieselben normalisierten Positionen über die Homographie im Videobild. Die y-Achse folgt den Rohkoordinaten und wird nicht nach Angriffsrichtung gespiegelt. Die Homographie modelliert die Spielfeldebene: Bei einem fliegenden Ball ist eine Abweichung zum sichtbaren Ball möglich.

In [ ]:
example_frame = int(frames.index[in_evaluation & ball_seen & frames["n_tracks"].ge(20).to_numpy()][0])
print(f"Beispiel: Video-Frame {example_frame} | Roh-Frame {frame_ids[example_frame]} | Videosekunde {example_frame / fps:.2f}")
print(f"Homographie: valid={homography.loc[example_frame, 'valid']}, reprojection_error={homography.loc[example_frame, 'reprojection_error']:.3f}")

In [ ]:
def read_video_frame(frame):
    capture = cv2.VideoCapture(str(paths["video"]))
    capture.set(cv2.CAP_PROP_POS_FRAMES, int(frame))
    ok, image = capture.read()
    capture.release()
    if not ok:
        raise RuntimeError(f"Video-Frame {frame} konnte nicht gelesen werden.")
    return cv2.cvtColor(image, cv2.COLOR_BGR2RGB)

def draw_pitch(ax):
    ax.set_facecolor("#edf5ec")
    ax.add_patch(Rectangle((0, 0), pitch_length, pitch_width, fill=False, edgecolor="0.4"))
    ax.axvline(pitch_length / 2, ymin=3 / (pitch_width + 6), ymax=(pitch_width + 3) / (pitch_width + 6), color="0.6", linewidth=1)
    ax.add_patch(Circle((pitch_length / 2, pitch_width / 2), 9.15, fill=False, edgecolor="0.6"))
    for x in [0, pitch_length - 16.5]:
        ax.add_patch(Rectangle((x, (pitch_width - 40.32) / 2), 16.5, 40.32, fill=False, edgecolor="0.6"))
    ax.set(xlim=(-3, pitch_length + 3), ylim=(pitch_width + 3, -3), aspect="equal", xlabel="x (m)", ylabel="y (m)")

In [ ]:
if homography.loc[example_frame, "valid"] != 1:
    raise ValueError("Für diesen Beispiel-Frame ist keine gültige Homographie vorhanden.")

image = read_video_frame(example_frame)
H = homography.loc[example_frame, h_columns].to_numpy(dtype=float).reshape(3, 3)
xy = values[example_frame, :, :2]
projected = np.c_[xy, np.ones(len(xy))] @ H.T
pixels = np.divide(projected[:, :2], projected[:, 2, None], out=np.full((len(xy), 2), np.nan), where=np.abs(projected[:, 2, None]) > 1e-10)

fig, axes = plt.subplots(1, 2, figsize=(15, 5), constrained_layout=True)
draw_pitch(axes[0])
axes[1].imshow(image)
for side in ["H", "A"]:
    mask = tracks["side"].eq(side).to_numpy()
    label = teams.set_index("side").loc[side, "team"]
    axes[0].scatter(xy[:-1][mask, 0] * pitch_length, xy[:-1][mask, 1] * pitch_width, s=45, color=TEAM_COLORS[side], label=label)
    axes[1].scatter(pixels[:-1][mask, 0], pixels[:-1][mask, 1], s=65, facecolors="none", edgecolors=TEAM_COLORS[side], linewidths=1.8, label=label)
    
axes[0].scatter(xy[-1, 0] * pitch_length, xy[-1, 1] * pitch_width, marker="*", s=100, color="#edb120", edgecolor="black", label="Ball")
axes[1].scatter(pixels[-1, 0], pixels[-1, 1], marker="*", s=100, color="#edb120", edgecolor="black", label="Ball (projiziert)")
axes[0].set_title(f"Tracking auf dem Spielfeld – Frame {example_frame}")
axes[0].legend(loc="upper center", bbox_to_anchor=(0.5, -0.15), ncol=3)
axes[1].set(title="Taktisches Video mit Trackingprojektion", xlim=(0, image.shape[1]), ylim=(image.shape[0], 0), xlabel="Pixel x", ylabel="Pixel y")
plt.show()

In [ ]:
active_tags = events.loc[(events["start_s"] <= example_frame / fps) & (events["end_s"] > example_frame / fps),
                         ["code", "Team", "Type", "start_s", "end_s"]]
print("Aktive Tags im Beispiel-Frame (Anzeige mit rechts offenem Intervall):")
display(active_tags)

**Sichtprüfung:** Liegen die Spielermarkierungen ungefähr an den Fußpunkten? Passt die Ballprojektion zur Szene? Falls nicht, sind Zeitversatz, Trackingfehler und Projektionseffekte getrennt zu prüfen. Ein passendes Einzelbild bestätigt nicht die Genauigkeit des gesamten Spiels. Die Tabellenprüfung oben bestätigt die Indexzuordnung, nicht die inhaltliche Richtigkeit der Trackingpositionen.

## 9. Aktueller Stand und mögliche nächste Schritte

Die Eingabedaten sind geladen und auf die taktische Videozeitachse bezogen. Die folgenden Kennzahlen werden aus den lokalen Dateien berechnet; sie sind **keine Modell- oder Evaluationsergebnisse**.

In [ ]:
display(Markdown(
    f"- **{n_frames:,} Frames** bei **{fps:g} fps**, davon **{in_evaluation.sum():,}** in den veröffentlichten Evaluationsbereichen.\n"
    f"- **{len(tracks)} Track-Slots** für zwei Teams; keine dauerhaften Spieler-IDs.\n"
    f"- Ballposition in **{100 * ball_seen[in_evaluation].mean():.1f} %** der bewerteten Frames beobachtet.\n"
    f"- **{len(events)} automatische Tag-Intervalle** und **{100 * homography['valid'].eq(1).mean():.1f} %** gültige Homographien im taktischen Video.\n"
    "- Keine Lückenfüllung, Entfernung von Beobachtungen, Besitzklassifikation oder Parameteroptimierung vorgenommen."
))

**Interpretation:** Die Ballposition ist ein wichtiger, aber unvollständiger Hinweis auf Kontrolle. Spieler-Slots und taktische Tags benötigen ebenfalls eine Qualitätsprüfung. Fehlendes Tracking und Spielunterbrechung sind unterschiedliche Sachverhalte.

**Noch offen:** Die öffentliche Ground Truth und die offiziellen Evaluationsskripte sind nicht Teil dieses Notebooks. Daher gibt es hier noch keinen Score. Die Geschwindigkeitseinheit des direkt gelesenen Rohkanals muss vor ihrer Nutzung geklärt werden.

**Vorschlag für den nächsten gemeinsam festzulegenden Schritt:** Die offizielle Baseline reproduzieren und auf dem öffentlichen Ausschnitt prüfen. Alternativ zunächst mehrere konkrete Szenen (Annahme, Zweikampf, Balllücke, Wiederanpfiff) visuell besprechen. Bestätigte Ballkontrolle wäre danach eine gezielte Erweiterung; ihre Regeln und Schwellen sind noch nicht festgelegt.

Für die spätere Abgabe zählt `spellF1` für ganze Ballbesitzphasen, mit `macroF1` als erstem Tie-Breaker. Die fünf öffentlichen Minuten mit 17 Ballbesitzphasen reichen zur Ausrichtungs- und Funktionsprüfung, aber nicht für belastbare umfangreiche Modellwahl. Details: [offizielle Evaluation](https://github.com/metrica-sports/more-than-a-hack-2027/blob/main/EVALUATION.md).